In [1]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 256, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [3]:
sentences1 = df["sentence1"].astype(str).tolist()
sentences2 = df["sentence2"].astype(str).tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
sentence_counts = Counter(all_sentences)
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
})

{'total_sentence_occurrences': 3000, 'num_unique_sentences': 2910, 'cache_hits': 90, 'cache_hit_rate': 0.03}


In [4]:
model = SentenceTransformer(model_name, device=device)
model.eval()

unique_embeddings = model.encode(
    unique_sentences,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print({
    "model_name": model_name,
    "embedding_shape": tuple(unique_embeddings.shape),
})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_shape': (2910, 384)}


In [5]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]].head(10))

                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  cosine_similarity  \
0      A man wearing a hard hat is dancing.  5.000           0.993372   
1                A child is riding a horse.  4.750           0.953962   
2  The man is feeding a mouse to the snake.  5.000           0.980101   
3                  A man is playing guitar.  2.400           0.647083   
4                 A man is playing a flute.  2.750           0.746228   
5                  A man is cutting onions.  2.615           0.754619   
6       The man

In [6]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

long_df = pd.concat([
    results_df[["sentence1", "predicted_score_0_5"]].rename(columns={"sentence1": "sentence"}),
    results_df[["sentence2", "predicted_score_0_5"]].rename(columns={"sentence2": "sentence"}),
], ignore_index=True)

repeated_sentence_stats = (
    long_df.groupby("sentence")
    .agg(
        occurrences=("sentence", "size"),
        min_predicted_score=("predicted_score_0_5", "min"),
        max_predicted_score=("predicted_score_0_5", "max"),
        mean_predicted_score=("predicted_score_0_5", "mean"),
    )
    .reset_index()
)
repeated_sentence_stats = repeated_sentence_stats[repeated_sentence_stats["occurrences"] > 1].copy()
repeated_sentence_stats["score_range"] = (
    repeated_sentence_stats["max_predicted_score"] - repeated_sentence_stats["min_predicted_score"]
)
repeated_sentence_stats = repeated_sentence_stats.sort_values(
    by=["occurrences", "score_range", "sentence"], ascending=[False, False, True]
).reset_index(drop=True)

print(repeated_sentence_stats.head(10))

                                     sentence  occurrences  \
0                  A man is playing a guitar.           13   
1                   A man is playing a flute.            7   
2                    A man is playing guitar.            6   
3                           A man is dancing.            5   
4                          A man is speaking.            4   
5                A woman is peeling a potato.            3   
6     Chinese shares close lower on Wednesday            3   
7   A group of people eat at a table outside.            3   
8                  A woman is riding a horse.            3   
9  A woman posing by a pillar with a U2 sign.            2   

   min_predicted_score  max_predicted_score  mean_predicted_score  score_range  
0             2.260297             4.947527              3.496032     2.687230  
1             2.642982             4.365569              3.594769     1.722587  
2             3.658068             4.939960              3.996560     1.28

In [7]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"num_repeated_sentences: {len(repeated_sentence_stats)}")
print("most_repeated_sentences_with_score_ranges:")
print(repeated_sentence_stats.head(10).to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")

device_used: mps
model_name: sentence-transformers/all-MiniLM-L6-v2
dataset_split: glue/stsb/validation
num_examples: 1500
total_sentence_occurrences: 3000
num_unique_sentences: 2910
cache_hits: 90
cache_hit_rate: 0.030000
pearson_correlation: 0.869619
spearman_correlation: 0.867164
num_repeated_sentences: 61
most_repeated_sentences_with_score_ranges:
[{'sentence': 'A man is playing a guitar.', 'occurrences': 13, 'min_predicted_score': 2.2602972984313965, 'max_predicted_score': 4.947526931762695, 'mean_predicted_score': 3.4960317611694336, 'score_range': 2.687229633331299}, {'sentence': 'A man is playing a flute.', 'occurrences': 7, 'min_predicted_score': 2.642981767654419, 'max_predicted_score': 4.3655686378479, 'mean_predicted_score': 3.594769239425659, 'score_range': 1.7225868701934814}, {'sentence': 'A man is playing guitar.', 'occurrences': 6, 'min_predicted_score': 3.6580681800842285, 'max_predicted_score': 4.939960479736328, 'mean_predicted_score': 3.9965600967407227, 'score_ran